# Dataset Management in RagaAI Catalyst

This notebook demonstrates how to manage datasets in **RagaAI Catalyst** (SDK v2.1.4.1). Datasets serve as the foundation for evaluating Retrieval-Augmented Generation (RAG) applications, analyzing trace metrics, and running benchmarking experiments.

### Table of Contents
1. [Setup and Environment Configuration](#1.-Setup-and-Environment-Configuration)
2. [Preparing a Sample Evaluation Dataset](#2.-Preparing-a-Sample-Evaluation-Dataset)
3. [Understanding and Defining Schema Mapping](#3.-Understanding-and-Defining-Schema-Mapping)
4. [Authenticating with RagaAI Catalyst](#4.-Authenticating-with-RagaAI-Catalyst)
5. [Initializing Dataset Management for a Project](#5.-Initializing-Dataset-Management-for-a-Project)
6. [Retrieving Supported Project Schema Elements](#6.-Retrieving-Supported-Project-Schema-Elements)
7. [Listing Existing Datasets](#7.-Listing-Existing-Datasets)
8. [Uploading and Creating a Dataset from CSV](#8.-Uploading-and-Creating-a-Dataset-from-CSV)
9. [Inspecting Dataset Columns](#9.-Inspecting-Dataset-Columns)
10. [Error Handling and Troubleshooting](#10.-Error-Handling-and-Troubleshooting)
11. [Summary and Next Steps](#11.-Summary-and-Next-Steps)

## 1. Setup and Environment Configuration

First, we configure our Python environment and import required packages.

> **Dependencies & SDK Architecture:**
> - **Dataset operations directly require**: `requests` (for REST API communication) and `pandas` (for CSV handling and tabular verification), alongside `python-dotenv` (for loading environment credentials).
> - **Full SDK Scope**: The full `ragaai_catalyst` package has broader installation requirements (including OpenTelemetry, LiteLLM, Groq, Google Generative AI, PyPDF2, etc.) because the SDK's top-level `__init__.py` exposes multiple platform modules (`Tracer`, `SyntheticDataGeneration`, `Evaluation`, and `GuardrailsManager`).
> - **Security & System Policy Troubleshooting**: If importing `ragaai_catalyst` encounters an `ImportError` due to a native binary extension (such as `grpc` or `.pyd` libraries blocked by Windows Application Control or AppLocker policies), do not attempt to disable or bypass security controls. Consult your system administrator to ensure your virtual environment directory has the required execution permissions.

In [1]:
import os
import sys
from dotenv import load_dotenv
import pandas as pd

# Initialize SDK import error tracking
SDK_IMPORT_ERROR = None

# Load environment variables from .env file before SDK imports
load_dotenv()

# Configure environment settings (use local fallback for model costs if offline)
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"

try:
    from ragaai_catalyst import RagaAICatalyst, Dataset
    print("RagaAI Catalyst SDK imported successfully!")
except ImportError as err:
    SDK_IMPORT_ERROR = err
    print(f"SDK Import Notice: {err}")
    print("If this is caused by an Application Control / AppLocker policy blocking a native")
    print("binary (such as gRPC or a C-extension), please consult your system administrator")
    print("to whitelist the virtual environment rather than disabling security controls.")

print(f"Pandas version: {pd.__version__}")

RagaAI Catalyst SDK imported successfully!
Pandas version: 2.2.3


## 2. Preparing a Sample Evaluation Dataset

RagaAI Catalyst evaluates RAG and LLM applications using datasets containing queries, model responses, retrieved contexts, and ground-truth expected responses.

Here, we locally generate a small, reproducible sample CSV file (`sample_rag_dataset.csv`) simulating queries and answers from an enterprise AI documentation assistant.

In [2]:
# Create a realistic sample evaluation dataset
sample_data = {
    "Query": [
        "What is RagaAI Catalyst?",
        "How do I track LLM traces with Catalyst?",
        "What metrics can be evaluated on datasets in Catalyst?",
        "How does schema mapping work for dataset uploads?",
        "Can I monitor guardrail violations in production?"
    ],
    "Response": [
        "RagaAI Catalyst is an enterprise platform for evaluating, tracing, and safeguarding LLM applications.",
        "You can use the Tracer class with auto-instrumentation or custom decorators to record traces.",
        "Supported metrics include Faithfulness, Hallucination, Response Latency, and custom criteria.",
        "Schema mapping connects CSV column names to platform elements like prompt, context, and response.",
        "Yes, GuardrailsManager and GuardExecutor validate and intercept runtime responses against predefined rules."
    ],
    "Context": [
        "RagaAI Catalyst provides dataset management, evaluation, tracing, and guardrails for LLM systems.",
        "The Catalyst Tracer supports automatic instrumentation for LangChain, OpenAI, and LlamaIndex.",
        "Evaluation experiments allow configuring LLM metrics and thresholds to assess dataset outputs.",
        "Dataset.create_from_csv requires a schema mapping dictionary matching columns to expected schema elements.",
        "GuardrailsManager lets users configure guardrails like regex checks and response evaluators on deployments."
    ],
    "ExpectedResponse": [
        "RagaAI Catalyst is a platform for evaluating, tracing, and safeguarding LLM applications.",
        "By using the Tracer class to capture traces and spans during execution.",
        "Metrics such as Faithfulness, Hallucination, and latency can be benchmarked.",
        "It maps CSV columns to schema elements such as prompt, context, response, and expected_response.",
        "Yes, GuardrailsManager manages guardrail configurations and GuardExecutor executes runtime checks."
    ]
}

# Convert dictionary to DataFrame and save to CSV
df_sample = pd.DataFrame(sample_data)
csv_file_path = "sample_rag_dataset.csv"
df_sample.to_csv(csv_file_path, index=False)

print(f"Sample dataset saved to: {csv_file_path}")
print(f"Dataset shape: {df_sample.shape[0]} rows, {df_sample.shape[1]} columns")
df_sample.head()

Sample dataset saved to: sample_rag_dataset.csv
Dataset shape: 5 rows, 4 columns


## 3. Understanding and Defining Schema Mapping

When creating a dataset from a CSV file, RagaAI Catalyst requires a `schema_mapping` dictionary. This mapping translates your custom CSV column headers into the standardized schema elements expected by Catalyst evaluation engines:

| CSV Column Header | Standard Schema Element | Purpose |
| :--- | :--- | :--- |
| `Query` | `prompt` | The user prompt or inquiry input |
| `Response` | `response` | The output produced by the LLM system |
| `Context` | `context` | Retrieved context or knowledge snippets |
| `ExpectedResponse` | `expected_response` | Ground-truth reference for accuracy/faithfulness benchmarks |

We can validate our local schema mapping against our CSV columns before uploading.

In [3]:
# Define schema mapping: {CSV_Column_Name: Schema_Element_Name}
schema_mapping = {
    "Query": "prompt",
    "Response": "response",
    "Context": "context",
    "ExpectedResponse": "expected_response"
}

# Local verification of columns before upload
csv_columns = list(pd.read_csv(csv_file_path).columns)
missing_columns = [col for col in schema_mapping.keys() if col not in csv_columns]

if not missing_columns:
    print("Schema mapping verification successful! All mapped columns exist in the CSV file.")
    for col, schema_elem in schema_mapping.items():
        print(f"  Column '{col}' -> Schema element '{schema_elem}'")
else:
    print(f"Validation error: Missing columns in CSV: {missing_columns}")

Schema mapping verification successful! All mapped columns exist in the CSV file.
  Column 'Query' -> Schema element 'prompt'
  Column 'Response' -> Schema element 'response'
  Column 'Context' -> Schema element 'context'
  Column 'ExpectedResponse' -> Schema element 'expected_response'


---
## 4. Authenticating with RagaAI Catalyst

> **Authentication Required:**
> The following cells interact directly with the Catalyst platform. You must provide your Catalyst credentials (`access_key` and `secret_key`) via environment variables or a `.env` file.
> 
> ### Recommended Cross-Platform Approach (`.env` file)
> Place a `.env` file in your working directory (ensure `.env` is listed in `.gitignore`):
> ```env
> RAGAAI_CATALYST_ACCESS_KEY=your_access_key_here
> RAGAAI_CATALYST_SECRET_KEY=your_secret_key_here
> RAGAAI_CATALYST_BASE_URL=https://your-catalyst-endpoint/api
> ```
> 
> ### Shell Environment Variables
> **Windows (PowerShell):**
> ```powershell
> $env:RAGAAI_CATALYST_ACCESS_KEY="your_access_key_here"
> $env:RAGAAI_CATALYST_SECRET_KEY="your_secret_key_here"
> $env:RAGAAI_CATALYST_BASE_URL="https://your-catalyst-endpoint/api"
> ```
> 
> **Linux / macOS (Bash):**
> ```bash
> export RAGAAI_CATALYST_ACCESS_KEY="your_access_key_here"
> export RAGAAI_CATALYST_SECRET_KEY="your_secret_key_here"
> export RAGAAI_CATALYST_BASE_URL="https://your-catalyst-endpoint/api"
> ```
> 
> > **Important Endpoint Notice:**
> > Do not assume `https://catalyst.raga.ai/api` is universally accessible or active for your deployment. The exact base URL depends on your organization's deployment (SaaS, dedicated cluster, or private cloud instance). Obtain the verified current base URL and credentials directly from your Catalyst administrator or current deployment documentation.

In [ ]:
# Note: This cell requires active Catalyst credentials and network connectivity to execute.
if SDK_IMPORT_ERROR is not None:
    print(f"Skipping authentication: SDK is unavailable due to import error ({SDK_IMPORT_ERROR}).")
else:
    access_key = os.getenv("RAGAAI_CATALYST_ACCESS_KEY")
    secret_key = os.getenv("RAGAAI_CATALYST_SECRET_KEY")
    base_url = os.getenv("RAGAAI_CATALYST_BASE_URL")

    # Validate credentials and base URL (never print or expose secret values)
    if not access_key or not secret_key:
        print("Notice: RAGAAI_CATALYST_ACCESS_KEY or RAGAAI_CATALYST_SECRET_KEY is not set.")
        print("Please configure your credentials before running authenticated cells.")
    elif not base_url:
        print("Notice: RAGAAI_CATALYST_BASE_URL is not set.")
        print("Please set your deployment base URL obtained from your Catalyst administrator.")
    else:
        # Initialize Catalyst client
        catalyst = RagaAICatalyst(
            access_key=access_key,
            secret_key=secret_key,
            base_url=base_url
        )
        print("RagaAICatalyst client initialized successfully!")

## 5. Initializing Dataset Management for a Project

The `Dataset` class manages all dataset operations scoped to a specific project. When instantiated, it verifies the existence of the project on Catalyst and retrieves its project ID.

In [ ]:
# Note: This cell requires active Catalyst credentials and an existing project on the platform.
if SDK_IMPORT_ERROR is not None or "Dataset" not in globals():
    print("Skipping project dataset initialization: RagaAI Catalyst SDK is not available.")
else:
    PROJECT_NAME = os.getenv("RAGAAI_PROJECT_NAME", "my_evaluation_project")

    try:
        dataset_manager = Dataset(project_name=PROJECT_NAME)
        print(f"Dataset manager initialized for project: {PROJECT_NAME}")
        print(f"Project ID: {dataset_manager.project_id}")
    except ValueError as e:
        print(f"Project verification error: {e}")
    except Exception as e:
        print(f"Initialization error: {e}")

## 6. Retrieving Supported Project Schema Elements

You can query the Catalyst backend to inspect the valid schema elements supported by the platform for dataset columns using `get_schema_mapping()`.

In [ ]:
# Note: This cell requires active Catalyst credentials.
if SDK_IMPORT_ERROR is not None or "dataset_manager" not in globals():
    print("Skipping schema mapping query: dataset_manager has not been initialized.")
else:
    try:
        supported_schema_elements = dataset_manager.get_schema_mapping()
        print("Supported schema elements on Catalyst:")
        for elem in supported_schema_elements:
            print(f"  - {elem}")
    except Exception as e:
        print(f"Failed to fetch schema elements: {e}")

## 7. Listing Existing Datasets

Before creating a new dataset, you can list the datasets already present in the project using `list_datasets()`.

In [ ]:
# Note: This cell requires active Catalyst credentials.
if SDK_IMPORT_ERROR is not None or "dataset_manager" not in globals():
    print("Skipping list datasets query: dataset_manager has not been initialized.")
else:
    try:
        existing_datasets = dataset_manager.list_datasets()
        print(f"Existing datasets in project '{PROJECT_NAME}':")
        if existing_datasets:
            for ds in existing_datasets:
                print(f"  - {ds}")
        else:
            print("  (No datasets found in this project)")
    except Exception as e:
        print(f"Failed to list datasets: {e}")

## 8. Uploading and Creating a Dataset from CSV

The `create_from_csv()` method performs the complete upload flow:
1. Validates that the dataset name is unique within the project.
2. Obtains a secure presigned upload URL from the Catalyst backend.
3. Performs a binary upload of the CSV file to cloud storage.
4. Registers the dataset with its associated `schema_mapping`.

Let's upload our `sample_rag_dataset.csv`.

In [ ]:
# Note: This cell requires active Catalyst credentials.
if SDK_IMPORT_ERROR is not None or "dataset_manager" not in globals():
    print("Skipping dataset upload: dataset_manager has not been initialized.")
else:
    from datetime import datetime

    # Generate a unique dataset name using a timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    new_dataset_name = f"rag_evaluation_dataset_{timestamp}"

    try:
        print(f"Uploading '{csv_file_path}' as dataset: '{new_dataset_name}'...")
        dataset_manager.create_from_csv(
            csv_path=csv_file_path,
            dataset_name=new_dataset_name,
            schema_mapping=schema_mapping
        )
        print(f"Dataset '{new_dataset_name}' created successfully!")
    except Exception as e:
        print(f"Failed to create dataset: {e}")

## 9. Inspecting Dataset Columns

After uploading, you can verify the registered columns of the dataset using `get_dataset_columns(dataset_name)`. This confirms that your CSV columns were successfully mapped and indexed.

In [ ]:
# Note: This cell requires active Catalyst credentials.
if SDK_IMPORT_ERROR is not None or "dataset_manager" not in globals() or "new_dataset_name" not in globals():
    print("Skipping dataset columns inspection: dataset_manager or new_dataset_name is not available.")
else:
    try:
        dataset_cols = dataset_manager.get_dataset_columns(dataset_name=new_dataset_name)
        print(f"Columns for dataset '{new_dataset_name}':")
        for col in dataset_cols:
            print(f"  - {col}")
    except Exception as e:
        print(f"Error fetching dataset columns: {e}")

## 10. Error Handling and Troubleshooting

Understanding SDK error handling prevents runtime failures during automated evaluation workflows.

### 1. Duplicate Dataset Name
Catalyst requires dataset names to be unique within a project. Calling `create_from_csv` with an existing name raises a `ValueError`.

In [ ]:
# Note: This cell requires active Catalyst credentials.
if SDK_IMPORT_ERROR is not None or "dataset_manager" not in globals() or "new_dataset_name" not in globals():
    print("Skipping duplicate dataset error demonstration: dataset_manager or new_dataset_name is not available.")
else:
    try:
        print("Attempting to re-upload with an existing dataset name...")
        dataset_manager.create_from_csv(
            csv_path=csv_file_path,
            dataset_name=new_dataset_name,  # Same dataset name as above
            schema_mapping=schema_mapping
        )
    except ValueError as e:
        print(f"Caught expected ValueError: {e}")
    except Exception as e:
        print(f"Error: {e}")

### 2. Local File-Existence Pre-Check
Before initiating an upload to Catalyst (which requests a presigned URL from the remote API), applications should perform a local pre-check to verify that the target CSV file exists on disk.

In [4]:
# Demonstrating local file existence pre-check prior to remote upload
csv_path_to_check = "non_existent_dataset.csv"

if not os.path.exists(csv_path_to_check):
    print(f"Pre-check validation: Local file not found: '{csv_path_to_check}'. Aborting upload before remote API request.")
else:
    print(f"File verified: '{csv_path_to_check}' is ready for upload.")

Pre-check validation: Local file not found: 'non_existent_dataset.csv'. Aborting upload before remote API request.


### 3. Non-existent Dataset Query
Calling `get_dataset_columns()` for a dataset that does not exist in the project raises a `ValueError`.

In [ ]:
# Note: This cell requires active Catalyst credentials.
if SDK_IMPORT_ERROR is not None or "dataset_manager" not in globals():
    print("Skipping non-existent dataset query: dataset_manager has not been initialized.")
else:
    try:
        dataset_manager.get_dataset_columns(dataset_name="non_existent_dataset_xyz")
    except ValueError as e:
        print(f"Caught expected ValueError: {e}")
    except Exception as e:
        print(f"Error: {e}")

## 11. Summary and Next Steps

In this notebook, we demonstrated the complete Dataset Management workflow in RagaAI Catalyst:
- **Local Preparation**: Creating structured CSV datasets containing query, response, context, and expected ground truth.
- **Schema Mapping**: Mapping arbitrary CSV column headers to Catalyst standard schema elements (`prompt`, `response`, `context`, `expected_response`).
- **Platform Operations**: Initializing the project-scoped `Dataset` manager, querying supported schema elements, uploading CSVs via presigned URLs, and inspecting indexed dataset columns.
- **Robust Error Handling**: Handling duplicate dataset names, local file pre-checks, and non-existent dataset queries.

### Next Steps: Running Evaluations
Once your dataset is uploaded, you can use it to run automated LLM evaluations and benchmarks using the `Evaluation` module (verified against `evaluation.py`):

```python
from ragaai_catalyst import Evaluation

# 1. Initialize Evaluation experiment for the uploaded dataset
evaluation = Evaluation(
    project_name=PROJECT_NAME,
    dataset_name=new_dataset_name
)

# 2. List available evaluation metrics supported by the platform
available_metrics = evaluation.list_metrics()
print("Available evaluation metrics:", available_metrics)

# 3. Add and configure evaluation metrics
# Note: Each metric requires 'name', 'config', 'column_name', and 'schema_mapping'
evaluation.add_metrics(
    metrics=[
        {
            "name": "Faithfulness",
            "config": {
                "model": "gpt-4o-mini",
                "provider": "openai",
                "threshold": {"gte": 0.5}
            },
            "column_name": "Faithfulness_v1",
            "schema_mapping": schema_mapping
        }
    ]
)

# 4. Monitor the status of the evaluation job
evaluation.get_status()

# 5. Retrieve the evaluated results DataFrame once completed
results_df = evaluation.get_results()
print(results_df.head())
```